In [1]:
!pip install inseq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.6/186.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 134.8 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.4
    Uninstalling typeguard-4.4.4:
      Successfully uninstalled typeguard-4.4.4
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which i

In [1]:
!pip install accelerate

In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 38.0 MB/s eta 0:00:00


In [23]:
# import packages
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
import json
import pandas as pd
from datasets import load_dataset
import torch
import inseq
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import numpy as np

In [4]:
#load model and tokenizer
model_name = "Qwen/Qwen2-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
#create a config object
config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
    ]
)
lora_model = get_peft_model(base_model, config)

In [6]:
#load the dataset and change its format
dataset = load_dataset(
    "json",
    data_files="/content/data.json"
)
#change the misconception id to label (MaE01 -> 1)
all_ids = sorted({ex["Misconception ID"] for split in dataset.values() for ex in split})
id2label = {mid: idx+1 for idx, mid in enumerate(all_ids)}

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
#change data to the format of text and label
def to_text_label(example):
    text = (
        f"Misconception: {example['Misconception']}."
        f"Topic: {example['Topic']}."
        f"Question: {example['Question']}."
        f"Incorrect Answer: {example['Incorrect Answer']}."
        f"Correct answer: {example['Correct Answer']}."
    )
    label = id2label[example["Misconception ID"]]
    return {"text": text, "label": label}

In [8]:
#map to the dataset
dataset = dataset["train"].map(to_text_label)
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in ["text","label"]])

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

In [9]:
dataset[:5]

{'text': ["Misconception: when students don't understand how to represent proportional relationships..Topic: Number sense.Question: What part is shaded?\nWrite a Fraction\n(Exercise A).Incorrect Answer: 1/3.Correct answer: 1/4.",
  "Misconception: when students don't understand how to represent proportional relationships..Topic: Number sense.Question: What part is shaded?\nWrite a Fraction\n(Exercise C).Incorrect Answer: 2/1.Correct answer: 2/3.",
  "Misconception: when students don't understand how to represent proportional relationships..Topic: Number sense.Question: What part is shaded?\nWrite a Fraction\n(Exercise D).Incorrect Answer: 2/2.Correct answer: 1/2.",
  "Misconception: when students don't understand how to represent proportional relationships..Topic: Number sense.Question: What part is shaded?\nWrite a Fraction\n(Exercise E).Incorrect Answer: 3/1.Correct answer: 3/4.",
  'Misconception: Students misunderstand proportional relationships, not realizing parts must be equal i

In [10]:
#tokenized the training data
tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize(
        batch,
        tokenizer=tokenizer
    ):
      tokenized_input = tokenizer(
        batch["text"],
    )
      return tokenized_input

#map to dataset and split train and eval
tokenized_datasets = dataset.map(tokenize, batched=True)

split_dataset = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

In [11]:
#training arguments
args = TrainingArguments(
    output_dir="./lora-qwen",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    weight_decay=0.1,
    learning_rate=2e-4,
    fp16=False,
    report_to="none",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=1,
    save_total_limit=2,
)

trainer = Trainer(
    model=lora_model,
    tokenizer=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
trainer.train()

/tmp/ipython-input-3815587382.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
1,2.780200
2,2.782300
3,2.869200
4,2.616900
5,2.744700
6,2.893600
7,2.914600
8,2.569300
9,2.904300
10,2.695900


TrainOutput(global_step=176, training_loss=1.8280201161449605, metrics={'train_runtime': 233.8385, 'train_samples_per_second': 6.021, 'train_steps_per_second': 0.753, 'total_flos': 1619440308965376.0, 'train_loss': 1.8280201161449605, 'epoch': 8.0})

In [12]:
#save the LoRA Adapter
lora_model.save_pretrained("./lora-qwen-adapter")
tokenizer.save_pretrained("./lora-qwen-adapter")

('./lora-qwen-adapter/tokenizer_config.json',
 './lora-qwen-adapter/special_tokens_map.json',
 './lora-qwen-adapter/chat_template.jinja',
 './lora-qwen-adapter/vocab.json',
 './lora-qwen-adapter/merges.txt',
 './lora-qwen-adapter/added_tokens.json',
 './lora-qwen-adapter/tokenizer.json')

In [13]:
#load the adapter to base model and tokenizer

att_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-1.5B-Instruct", trust_remote_code=True)
if att_tokenizer.pad_token is None:
  att_tokenizer.add_special_tokens({'pad_token': att_tokenizer.eos_token})
if att_tokenizer.pad_token_id is None:
  att_tokenizer.pad_token_id = att_tokenizer.eos_token_id
att_tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-1.5B-Instruct",
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "./lora-qwen-adapter")
merged_model = model.merge_and_unload()
merged_model.resize_token_embeddings(len(att_tokenizer))
device = "cuda" if torch.cuda.is_available() else "cpu"
merged_model.to(device)


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151646, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [14]:
#check the fine-tuned model's attention
attribution_model = inseq.load_model(
    model = merged_model,
    tokenizer = att_tokenizer,
    attribution_method = "integrated_gradients",
    internal_batch_size = "model.layers.self_attn"
)
inseq_tokenizer = attribution_model.tokenizer
if inseq_tokenizer.pad_token is None:
  inseq_tokenizer.pad_token = inseq_tokenizer.eos_token
if inseq_tokenizer.pad_token_id is None:
    inseq_tokenizer.pad_token_id = inseq_tokenizer.eos_token_id
inseq_tokenizer.padding_side = "left"

In [15]:
#reload the dataset
dataset = load_dataset(
    "json",
    data_files="/content/data.json"
)

all_ids = sorted({ex["Misconception ID"] for split in dataset.values() for ex in split})
id2label = {mid: idx+1 for idx, mid in enumerate(all_ids)}

In [16]:
def to_text_label(example):
    text = (
        f"Topic: {example['Topic']}."
        f"Question: {example['Question']}."
        f"Incorrect Answer: {example['Incorrect Answer']}."
        f"Correct answer: {example['Correct Answer']}."
        f"Task: What is Misconception?"
    )
    label = id2label[example["Misconception ID"]]
    return {"text": text, "label": label}

In [17]:
dataset = dataset["train"].map(to_text_label)

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

In [19]:
#exmaple for test the integrated gradients of the input text to the corresponding label
example_a = {
        "Misconception":"when students simplify just one of the terms in a fraction, either the numerator or the denominator",
        "Misconception ID":"MaE07",
        "Topic":"Number Operations",
        "Example Number":1,
        "Question":"Simplify the fraction 4/8",
        "Incorrect Answer":"4/8=2/8",
        "Correct Answer":"4/8=1/4",
        "Question image":"",
        "Learner Answer image":"",
        "Correct Answer image":"",
        "Source":"Ashlock, 2006",
        "Explanation":"The learner simplified the numerator, but not the denominator"
    }

#map the data to the text and label
processed = to_text_label(example_a)
text = processed['text']
label = processed['label']

#tokenize the input text
inputs = inseq_tokenizer(text, return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to(merged_model.device)
attention_mask = inputs.attention_mask.to(merged_model.device)

generated_output = merged_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=20
)
generated_text = inseq_tokenizer.batch_decode(generated_output, skip_special_tokens=True)[0]

out = attribution_model.attribute(
    input_texts = text,
    targets= [generated_text],
    internal_batch_size=2,
    n_steps=10
)
out.show()

Attributing with integrated_gradients...: 100%|██████████| 66/66 [00:13<00:00,  1.53it/s]


,ĠThe,Ġstudent,Ġincorrectly,Ġsimplified,Ġthe,Ġfraction,Ġ,4,/,8,Ġto,Ġ,2,/,8,.,ĠTopic,:,ĠNumber,ĠOperations
Topic,0.073,0.07,0.078,0.089,0.079,0.063,0.049,0.088,0.057,0.073,0.048,0.065,0.044,0.042,0.036,0.061,0.095,0.03,0.099,0.083
:,0.049,0.037,0.029,0.026,0.028,0.026,0.025,0.028,0.02,0.025,0.018,0.022,0.019,0.018,0.014,0.026,0.049,0.017,0.044,0.033
ĠNumber,0.028,0.027,0.025,0.029,0.026,0.021,0.022,0.032,0.023,0.029,0.016,0.021,0.022,0.015,0.017,0.019,0.019,0.015,0.029,0.082
ĠOperations,0.038,0.041,0.03,0.036,0.031,0.029,0.027,0.041,0.028,0.037,0.025,0.027,0.023,0.017,0.021,0.029,0.021,0.016,0.031,0.369
.Question,0.114,0.084,0.171,0.164,0.12,0.093,0.062,0.17,0.128,0.15,0.061,0.054,0.056,0.07,0.077,0.093,0.12,0.04,0.066,0.057
:,0.026,0.03,0.019,0.024,0.022,0.02,0.02,0.029,0.02,0.026,0.016,0.019,0.015,0.03,0.027,0.023,0.024,0.016,0.018,0.01
ĠSimpl,0.018,0.015,0.017,0.032,0.014,0.018,0.017,0.013,0.021,0.021,0.022,0.038,0.028,0.075,0.067,0.038,0.011,0.016,0.016,0.007
ify,0.009,0.006,0.009,0.016,0.009,0.01,0.01,0.007,0.011,0.007,0.012,0.012,0.013,0.015,0.007,0.006,0.007,0.01,0.007,0.004
Ġthe,0.008,0.007,0.007,0.006,0.01,0.009,0.011,0.007,0.009,0.008,0.01,0.011,0.015,0.015,0.007,0.008,0.007,0.009,0.006,0.003
Ġfraction,0.018,0.012,0.008,0.01,0.015,0.024,0.014,0.013,0.019,0.012,0.016,0.02,0.02,0.03,0.014,0.014,0.01,0.007,0.008,0.005


In [30]:
generated_text

'Topic: Number Operations.Question: Simplify the fraction 4/8.Incorrect Answer: 4/8=2/8.Correct answer: 4/8=1/4.Task: What is Misconception? How can I correct it?\nThe student thought that 4/8 = 2/8,'

In [31]:
#exmaple for test the integrated gradients of the input text to the corresponding label
example_b = {
        "Misconception":"when students wrongly divide fractions by splitting numerators and denominators into separate divisions, ignoring remainders",
        "Misconception ID":"MaE14",
        "Topic":"Number Operations",
        "Example Number":4,
        "Question":"7/5u00f73/2=?",
        "Incorrect Answer":"7/5u00f73/2=2/2",
        "Correct Answer":"14/15",
        "Question image":"",
        "Learner Answer image":"",
        "Correct Answer image":"",
        "Source":"Ashlock (2006)np. 174",
        "Explanation":""
    }

#map the data to the text and label
processed = to_text_label(example_b)
text = processed['text']
label = processed['label']

#tokenize the input text
inputs = inseq_tokenizer(text, return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to(merged_model.device)
attention_mask = inputs.attention_mask.to(merged_model.device)

generated_output = merged_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=20
)
generated_text = inseq_tokenizer.batch_decode(generated_output, skip_special_tokens=True)[0]

out = attribution_model.attribute(
    input_texts = text,
    targets= [generated_text],
    internal_batch_size=2,
    n_steps=10
)
out.show()


Attributing with integrated_gradients...: 100%|██████████| 77/77 [00:15<00:00,  1.33it/s]


,Number,Ġoperations,.Ċ,Question,:,Ġ,7,/,5,u,0,0,f,7,3,/,2,=?,Incorrect,Ġanswer
Topic,0.084,0.07,0.061,0.09,0.03,0.088,0.09,0.065,0.054,0.054,0.042,0.042,0.034,0.044,0.049,0.058,0.024,0.049,0.045,0.045
:,0.043,0.04,0.03,0.045,0.013,0.036,0.045,0.031,0.025,0.033,0.028,0.015,0.017,0.016,0.019,0.023,0.01,0.031,0.023,0.024
ĠNumber,0.032,0.045,0.037,0.026,0.017,0.033,0.031,0.036,0.027,0.038,0.038,0.019,0.023,0.026,0.019,0.033,0.01,0.025,0.023,0.022
ĠOperations,0.041,0.093,0.049,0.034,0.018,0.058,0.029,0.038,0.028,0.057,0.065,0.027,0.03,0.037,0.023,0.041,0.011,0.028,0.032,0.034
.Question,0.099,0.08,0.083,0.13,0.029,0.108,0.111,0.094,0.092,0.121,0.088,0.039,0.062,0.049,0.062,0.071,0.023,0.096,0.086,0.066
:,0.04,0.024,0.024,0.028,0.013,0.033,0.046,0.031,0.019,0.025,0.019,0.011,0.017,0.013,0.013,0.016,0.011,0.027,0.022,0.021
Ġ,0.015,0.011,0.007,0.01,0.011,0.015,0.027,0.015,0.012,0.022,0.019,0.013,0.018,0.011,0.012,0.016,0.013,0.013,0.013,0.011
7,0.005,0.005,0.007,0.004,0.007,0.011,0.006,0.013,0.009,0.014,0.013,0.011,0.013,0.009,0.007,0.015,0.013,0.007,0.005,0.004
/,0.005,0.004,0.008,0.005,0.005,0.011,0.005,0.02,0.012,0.017,0.021,0.012,0.025,0.016,0.008,0.024,0.007,0.013,0.009,0.004
5,0.006,0.005,0.009,0.005,0.015,0.015,0.006,0.012,0.015,0.021,0.032,0.026,0.042,0.03,0.014,0.02,0.014,0.02,0.007,0.004


In [32]:
generated_text

"Topic: Number Operations.Question: 7/5u00f73/2=?.Incorrect Answer: 7/5u00f73/2=2/2.Correct answer: 14/15.Task: What is Misconception? (1) Division of fractions; (2) division of whole numbers and fractions.\nThe student's"

In [33]:
example_c = {
        "Misconception":"when students struggle to understand that ratios can compare same or different units",
        "Misconception ID":"MaE24",
        "Topic":"Ratios and proportional reasoning",
        "Example Number":1,
        "Question":"To bake donuts, Mariah needs 8 cups of flour to bake 14 donuts. Using the same recipe, how many donuts can she bake with 12 cups of flour?",
        "Incorrect Answer":"I am counting... I am trying to divide 14 by 8... I got 1.75 and I times (multiplied) by 12.\n1.75*12 you get... 350+175, so... 35.0+17.5=53\n53 donuts",
        "Correct Answer":"In this case, it is intended to find the unit value to relate donuts to cups of flour, then calculate the amount needed with the proportional method, observing the correct units involved:\n(14 donuts)\/(8 cups)=(1.75 donuts)\/(1 cup)\n(1.75 donuts\/cup)*(12 cups)=21 donuts\n21 donuts",
        "Question image":"",
        "Learner Answer image":"",
        "Correct Answer image":"",
        "Source":"Singh, 2000\np. 16-17",
        "Explanation":"She did not try to construct a relationship between 8 cups and 12 donuts as 1 1\/2 times more, which is an important criterion in multiplicative reasoning. She relied heavily on  algorithmic procedures and utilized additive reasoning"
    }

#map the data to the text and label
processed = to_text_label(example_c)
text = processed['text']
label = processed['label']

#tokenize the input text
inputs = inseq_tokenizer(text, return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to(merged_model.device)
attention_mask = inputs.attention_mask.to(merged_model.device)

generated_output = merged_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=20
)
generated_text = inseq_tokenizer.batch_decode(generated_output, skip_special_tokens=True)[0]

out = attribution_model.attribute(
    input_texts = text,
    targets= [generated_text],
    internal_batch_size=2,
    n_steps=10
)
out.show()

<>:8: SyntaxWarning: invalid escape sequence '\/'
<>:13: SyntaxWarning: invalid escape sequence '\/'
<>:8: SyntaxWarning: invalid escape sequence '\/'
<>:13: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipython-input-1847557374.py:8: SyntaxWarning: invalid escape sequence '\/'
  "Correct Answer":"In this case, it is intended to find the unit value to relate donuts to cups of flour, then calculate the amount needed with the proportional method, observing the correct units involved:\n(14 donuts)\/(8 cups)=(1.75 donuts)\/(1 cup)\n(1.75 donuts\/cup)*(12 cups)=21 donuts\n21 donuts",
/tmp/ipython-input-1847557374.py:13: SyntaxWarning: invalid escape sequence '\/'
  "Explanation":"She did not try to construct a relationship between 8 cups and 12 donuts as 1 1\/2 times more, which is an important criterion in multiplicative reasoning. She relied heavily on  algorithmic procedures and utilized additive reasoning"
Attributing with integrated_gradients...: 100%|██████████| 242/242 [00:40<00:0

,The,Ġproblem,Ġwas,Ġtoo,Ġeasy,Ġfor,Ġme,",",ĠI,Ġdidn,'t,Ġneed,Ġto,Ġuse,Ġthe,Ġproportion,Ġmethod,Ġat,Ġall,","
Topic,0.044,0.039,0.046,0.037,0.028,0.03,0.039,0.02,0.022,0.038,0.024,0.021,0.019,0.015,0.021,0.028,0.013,0.024,0.015,0.017
:,0.022,0.022,0.023,0.018,0.013,0.01,0.019,0.013,0.008,0.019,0.01,0.009,0.008,0.008,0.01,0.009,0.006,0.013,0.005,0.009
ĠRat,0.012,0.009,0.007,0.006,0.005,0.006,0.006,0.006,0.004,0.005,0.006,0.004,0.005,0.008,0.008,0.008,0.006,0.004,0.004,0.004
ios,0.006,0.007,0.006,0.006,0.005,0.004,0.005,0.004,0.003,0.003,0.004,0.003,0.006,0.007,0.01,0.005,0.006,0.005,0.004,0.003
Ġand,0.005,0.01,0.006,0.006,0.006,0.005,0.005,0.005,0.004,0.006,0.004,0.003,0.003,0.005,0.006,0.007,0.004,0.007,0.003,0.004
Ġproportional,0.009,0.021,0.014,0.016,0.016,0.01,0.012,0.008,0.008,0.009,0.007,0.007,0.012,0.012,0.038,0.034,0.021,0.013,0.008,0.009
Ġreasoning,0.015,0.024,0.019,0.018,0.018,0.016,0.015,0.013,0.013,0.017,0.012,0.011,0.01,0.013,0.014,0.016,0.012,0.016,0.007,0.012
.Question,0.091,0.043,0.09,0.053,0.087,0.039,0.054,0.047,0.034,0.055,0.039,0.045,0.023,0.02,0.015,0.035,0.012,0.052,0.023,0.027
:,0.014,0.014,0.017,0.011,0.015,0.008,0.013,0.013,0.008,0.012,0.009,0.008,0.007,0.005,0.005,0.007,0.003,0.01,0.004,0.011
ĠTo,0.011,0.005,0.006,0.004,0.004,0.003,0.004,0.006,0.004,0.004,0.003,0.003,0.003,0.004,0.002,0.003,0.002,0.003,0.003,0.004


In [34]:
generated_text

'Topic: Ratios and proportional reasoning.Question: To bake donuts, Mariah needs 8 cups of flour to bake 14 donuts. Using the same recipe, how many donuts can she bake with 12 cups of flour?.Incorrect Answer: I am counting... I am trying to divide 14 by 8... I got 1.75 and I times (multiplied) by 12.\n1.75*12 you get... 350+175, so... 35.0+17.5=53\n53 donuts.Correct answer: In this case, it is intended to find the unit value to relate donuts to cups of flour, then calculate the amount needed with the proportional method, observing the correct units involved:\n(14 donuts)\\/(8 cups)=(1.75 donuts)\\/(1 cup)\n(1.75 donuts\\/cup)*(12 cups)=21 donuts\n21 donuts.Task: What is Misconception?In this case, students are asked to apply a ratio-based proportional reasoning approach in order to determine the'